# Compare Multiple ASR Evaluation Runs (with Timing)

This notebook allows you to compare results from multiple evaluation runs with different configurations or parameters.

## Features:
- Load and compare multiple run results
- Visualize performance across runs
- Analyze parameter effects
- Generate comparison reports
- **NEW**: Timing and execution logging

---

## 1. Load Multiple Run Results

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import glob
import time

# Add scripts to path
import sys
sys.path.append('scripts')

# Import timing utilities
from utils.timing_utils import initialize_timing, log_cell, log_notebook_completion

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Initialize timing system
notebook_start_time = time.time()
timer = initialize_timing("results/execution_logs/comparison_timing.json")

print("🔍 Looking for previous runs...")

# Find all run directories
results_dir = Path("results/comparisons")
run_dirs = [d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith("run_")]
run_dirs.sort(reverse=True)  # Most recent first

print(f"Found {len(run_dirs)} previous runs:")
for i, run_dir in enumerate(run_dirs):
    print(f"  {i+1}. {run_dir.name}")

if len(run_dirs) == 0:
    print("❌ No previous runs found. Please run main evaluation notebook first.")

## 2. Select Runs to Compare

In [ ]:
def load_comparison_data(selected_runs):
    """Load results from selected runs with timing"""
    with log_cell("Loading Comparison Data"):
        all_results = []
        run_configs = {}
        run_timing_data = {}
        
        for run_dir in selected_runs:
            print(f"  Loading {run_dir.name}...")
            
            # Load run metadata
            metadata_file = run_dir / f"run_metadata_{run_dir.name.replace('run_', '')}.json"
            
            if metadata_file.exists():
                with open(metadata_file, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)
                
                run_configs[run_dir.name] = metadata
                
                # Load timing data if available
                timing_file = Path(f"results/execution_logs/timing_log_{metadata['run_id']}.json")
                if timing_file.exists():
                    with open(timing_file, 'r', encoding='utf-8') as f:
                        timing_data = json.load(f)
                    run_timing_data[run_dir.name] = timing_data
                
                # Load results
                results_file = run_dir / f"asr_compare_{metadata['run_id']}.json"
                if results_file.exists():
                    with open(results_file, 'r', encoding='utf-8') as f:
                        run_results = json.load(f)
                    
                    # Add run information to each result
                    for result in run_results:
                        result['run_name'] = run_dir.name
                        result['run_timestamp'] = metadata['timestamp']
                        result['run_config'] = metadata.get('config', {})
                    
                    all_results.extend(run_results)
                    print(f"    ✅ Loaded {len(run_results)} results from {run_dir.name}")
                else:
                    print(f"    ❌ Results file not found for {run_dir.name}")
            else:
                print(f"    ❌ Metadata file not found for {run_dir.name}")
        
        print(f"\n📈 Total results loaded: {len(all_results)}")
        return all_results, run_configs, run_timing_data

# Select runs to compare (modify as needed)
# Options: 
# - Use all runs: selected_runs = run_dirs
# - Use specific runs: selected_runs = [run_dirs[0], run_dirs[1]]
# - Use N most recent: selected_runs = run_dirs[:3]

# Example: Compare 3 most recent runs
selected_runs = run_dirs[:3] if len(run_dirs) >= 3 else run_dirs

print(f"\n📊 Selected {len(selected_runs)} runs for comparison:")
for run_dir in selected_runs:
    print(f"  - {run_dir.name}")

# Load data
all_results, run_configs, run_timing_data = load_comparison_data(selected_runs)

## 3. Execution Timing Analysis

In [ ]:
def analyze_timing_data(run_timing_data):
    """Analyze timing data across runs"""
    with log_cell("Timing Data Analysis"):
        if not run_timing_data:
            print("❌ No timing data available for selected runs")
            return None
        
        print("\n⏱️  Execution Timing Analysis:")
        
        timing_summary = []
        
        for run_name, timing_log in run_timing_data.items():
            if not timing_log:
                continue
                
            # Calculate total time and operation breakdown
            total_time = sum(entry.get('elapsed_seconds', 0) for entry in timing_log if 'elapsed_seconds' in entry)
            
            # Extract operation times
            operations = {}
            for entry in timing_log:
                if 'operation' in entry and 'elapsed_seconds' in entry:
                    op_name = entry['operation']
                    if op_name not in operations:
                        operations[op_name] = []
                    operations[op_name].append(entry['elapsed_seconds'])
            
            # Average operation times
            avg_operations = {op: sum(times)/len(times) for op, times in operations.items()}
            
            timing_summary.append({
                'run_name': run_name,
                'total_time': total_time,
                'operation_count': len(timing_log),
                'operations': avg_operations
            })
            
            minutes, seconds = divmod(total_time, 60)
            print(f"\n{run_name}:")
            print(f"  Total time: {minutes:.0f}m {seconds:.1f}s")
            print(f"  Operations: {len(timing_log)}")
            
            # Show slowest operations
            if avg_operations:
                sorted_ops = sorted(avg_operations.items(), key=lambda x: x[1], reverse=True)
                print(f"  Slowest operations:")
                for op_name, avg_time in sorted_ops[:3]:
                    print(f"    - {op_name}: {avg_time:.1f}s")
        
        return timing_summary

# Analyze timing data
timing_summary = analyze_timing_data(run_timing_data)

## 4. Performance Comparison with Timing

In [ ]:
def create_comprehensive_comparison(all_results, run_configs, timing_summary):
    """Create comprehensive comparison with timing data"""
    with log_cell("Comprehensive Performance Comparison"):
        if not all_results:
            print("❌ No results to compare")
            return
        
        df = pd.DataFrame(all_results)
        
        # Create figure with timing subplot
        fig = plt.figure(figsize=(20, 16))
        gs = fig.add_gridspec(3, 3, height_ratios=[2, 2, 1], width_ratios=[1, 1, 1])
        
        fig.suptitle('ASR Performance Comparison Across Multiple Runs (with Timing)', fontsize=16, fontweight='bold')
        
        # 1. CER by Model and Run
        ax1 = fig.add_subplot(gs[0, 0])
        pivot_cer = df.pivot_table(values='cer', index='run_name', columns='model', aggfunc='mean')
        pivot_cer.plot(kind='bar', ax=ax1, width=0.8)
        ax1.set_title('CER by Model and Run')
        ax1.set_ylabel('Character Error Rate')
        ax1.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.tick_params(axis='x', rotation=45)
        
        # 2. Original vs Shortened Comparison
        ax2 = fig.add_subplot(gs[0, 1])
        condition_pivot = df.pivot_table(values='cer', index='run_name', columns='condition', aggfunc='mean')
        condition_pivot.plot(kind='bar', ax=ax2, width=0.8)
        ax2.set_title('Original vs Shortened Audio')
        ax2.set_ylabel('CER')
        ax2.legend(title='Condition')
        ax2.tick_params(axis='x', rotation=45)
        
        # 3. Execution Time Comparison
        ax3 = fig.add_subplot(gs[0, 2])
        if timing_summary:
            timing_df = pd.DataFrame(timing_summary)
            ax3.bar(range(len(timing_df)), timing_df['total_time'])
            ax3.set_title('Total Execution Time by Run')
            ax3.set_ylabel('Time (seconds)')
            ax3.set_xlabel('Run')
            ax3.set_xticks(range(len(timing_df)))
            ax3.set_xticklabels([name.replace('run_', '') for name in timing_df['run_name']], rotation=45)
        else:
            ax3.text(0.5, 0.5, 'No timing data available', ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title('Execution Time (No Data)')
        
        # 4. Error Type Comparison
        ax4 = fig.add_subplot(gs[1, 0])
        error_types = ['substitution_rate', 'insertion_rate', 'deletion_rate']
        error_summary = df.groupby('run_name')[error_types].mean()
        error_summary.plot(kind='bar', ax=ax4, stacked=True)
        ax4.set_title('Error Type Distribution by Run')
        ax4.set_ylabel('Error Rate')
        ax4.legend(title='Error Type')
        ax4.tick_params(axis='x', rotation=45)
        
        # 5. Model Performance Trend
        ax5 = fig.add_subplot(gs[1, 1])
        df_sorted = df.copy()
        df_sorted['run_datetime'] = pd.to_datetime(df_sorted['run_timestamp'])
        df_sorted = df_sorted.sort_values('run_datetime')
        
        for model in df_sorted['model'].unique():
            model_data = df_sorted[df_sorted['model'] == model]
            model_avg = model_data.groupby('run_name')['cer'].mean().reset_index()
            model_avg = model_avg.merge(df_sorted[['run_name', 'run_datetime']].drop_duplicates(), on='run_name')
            model_avg = model_avg.sort_values('run_datetime')
            ax5.plot(range(len(model_avg)), model_avg['cer'], marker='o', label=model, linewidth=2)
        
        ax5.set_title('Model Performance Trend Over Time')
        ax5.set_ylabel('CER')
        ax5.set_xlabel('Run Sequence')
        ax5.legend(title='Model')
        ax5.grid(True, alpha=0.3)
        
        # 6. Performance vs Time Efficiency
        ax6 = fig.add_subplot(gs[1, 2])
        if timing_summary:
            # Create performance vs efficiency scatter
            perf_eff_data = []
            for timing_info in timing_summary:
                run_name = timing_info['run_name']
                run_data = df[df['run_name'] == run_name]
                if len(run_data) > 0:
                    avg_cer = run_data['cer'].mean()
                    total_time = timing_info['total_time']
                    perf_eff_data.append({
                        'run_name': run_name,
                        'avg_cer': avg_cer,
                        'total_time': total_time
                    })
            
            if perf_eff_data:
                perf_df = pd.DataFrame(perf_eff_data)
                scatter = ax6.scatter(perf_df['total_time'], perf_df['avg_cer'], 
                                 s=100, alpha=0.7, c=range(len(perf_df)), cmap='viridis')
                ax6.set_xlabel('Total Execution Time (seconds)')
                ax6.set_ylabel('Average CER')
                ax6.set_title('Performance vs Time Efficiency')
                ax6.grid(True, alpha=0.3)
                
                # Add run labels
                for i, row in perf_df.iterrows():
                    ax6.annotate(row['run_name'].replace('run_', ''), 
                               (row['total_time'], row['avg_cer']), 
                               xytext=(5, 5), textcoords='offset points', fontsize=8)
        else:
            ax6.text(0.5, 0.5, 'No timing data available', ha='center', va='center', transform=ax6.transAxes)
            ax6.set_title('Performance vs Efficiency (No Data)')
        
        # 7. Timing Breakdown (if available)
        ax7 = fig.add_subplot(gs[2, :])
        if timing_summary and len(timing_summary) > 0:
            # Create operation timing comparison
            all_operations = set()
            for timing_info in timing_summary:
                all_operations.update(timing_info['operations'].keys())
            
            if all_operations:
                op_matrix = []
                for timing_info in timing_summary:
                    row = {'run_name': timing_info['run_name']}
                    for op in all_operations:
                        row[op] = timing_info['operations'].get(op, 0)
                    op_matrix.append(row)
                
                op_df = pd.DataFrame(op_matrix).set_index('run_name')
                op_df.plot(kind='bar', ax=ax7, width=0.8, stacked=True)
                ax7.set_title('Operation Timing Breakdown by Run')
                ax7.set_ylabel('Time (seconds)')
                ax7.legend(title='Operation', bbox_to_anchor=(1.05, 1), loc='upper left')
                ax7.tick_params(axis='x', rotation=45)
            else:
                ax7.text(0.5, 0.5, 'No operation data available', ha='center', va='center', transform=ax7.transAxes)
                ax7.set_title('Operation Timing (No Data)')
        else:
            ax7.text(0.5, 0.5, 'No timing data available', ha='center', va='center', transform=ax7.transAxes)
            ax7.set_title('Operation Timing (No Data)')
        
        plt.tight_layout()
        plt.show()
        
        # Save comparison plots
        comparison_plot_path = f"results/multi_run_comparisons/comparison_plots_with_timing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
        print(f"📊 Comparison plots saved to: {comparison_plot_path}")
        
        return comparison_plot_path

# Create comprehensive comparison
plot_path = create_comprehensive_comparison(all_results, run_configs, timing_summary)

## 5. Export Enhanced Comparison Report

In [ ]:
def export_enhanced_comparison_report(all_results, run_configs, timing_summary, plot_path):
    """Export comprehensive comparison report with timing data"""
    with log_cell("Export Enhanced Comparison Report"):
        if not all_results:
            print("❌ No results to export")
            return
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Calculate best performance metrics
        df = pd.DataFrame(all_results)
        best_overall = df.loc[df['cer'].idxmin()] if len(df) > 0 else None
        
        # Create comprehensive comparison report
        comparison_report = {
            "comparison_timestamp": datetime.now().isoformat(),
            "runs_compared": [run.name for run in selected_runs],
            "total_results": len(all_results),
            "run_configurations": run_configs,
            "timing_summary": timing_summary,
            "all_results": all_results,
            "analysis_summary": {
                "best_overall": {
                    "model": best_overall['model'] if best_overall is not None else None,
                    "condition": best_overall['condition'] if best_overall is not None else None,
                    "run_name": best_overall['run_name'] if best_overall is not None else None,
                    "cer": best_overall['cer'] if best_overall is not None else None
                },
                "timing_insights": {
                    "total_runs_with_timing": len(timing_summary) if timing_summary else 0,
                    "slowest_run": max(timing_summary, key=lambda x: x['total_time'])['run_name'] if timing_summary else None,
                    "fastest_run": min(timing_summary, key=lambda x: x['total_time'])['run_name'] if timing_summary else None
                }
            }
        }
        
        # Save comparison report
        report_path = f"results/multi_run_comparisons/comparison_report_with_timing_{timestamp}.json"
        
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(comparison_report, f, ensure_ascii=False, indent=2)
        
        # Save CSV summary with timing
        summary_df = pd.DataFrame(all_results)[
            ['run_name', 'run_timestamp', 'model', 'condition', 'cer', 
             'substitution_rate', 'insertion_rate', 'deletion_rate']
        ]
        
        # Add timing data if available
        if timing_summary:
            timing_df = pd.DataFrame(timing_summary)[['run_name', 'total_time']]
            summary_df = summary_df.merge(timing_df, on='run_name', how='left')
        
        csv_path = f"results/multi_run_comparisons/comparison_summary_with_timing_{timestamp}.csv"
        summary_df.to_csv(csv_path, index=False)
        
        print(f"\n📋 Enhanced Comparison Report Generated:")
        print(f"  📄 JSON Report: {report_path}")
        print(f"  📊 CSV Summary: {csv_path}")
        print(f"  📈 Plots: {plot_path}")
        
        # Create summary text file
        summary_text = f"""
ASR Multi-Run Comparison Report (with Timing)
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
========================================

Runs Compared: {len(selected_runs)}
Total Results: {len(all_results)}
Runs with Timing Data: {len(timing_summary) if timing_summary else 0}

Runs:
{chr(10).join([f'  - {run.name}' for run in selected_runs])}

Best Overall Performance:
  Model: {best_overall['model'] if best_overall is not None else 'N/A'}
  Condition: {best_overall['condition'] if best_overall is not None else 'N/A'}
  Run: {best_overall['run_name'] if best_overall is not None else 'N/A'}
  CER: {best_overall['cer']:.4f if best_overall is not None else 'N/A'}

Timing Insights:
  Runs with timing: {len(timing_summary) if timing_summary else 0}
  Fastest run: {min(timing_summary, key=lambda x: x['total_time'])['run_name'] if timing_summary else 'N/A'}
  Slowest run: {max(timing_summary, key=lambda x: x['total_time'])['run_name'] if timing_summary else 'N/A'}

Files Generated:
  - {report_path}
  - {csv_path}
  - {plot_path}
"""
        
        text_path = f"results/multi_run_comparisons/comparison_summary_with_timing_{timestamp}.txt"
        with open(text_path, 'w', encoding='utf-8') as f:
            f.write(summary_text)
        
        print(f"  📝 Text Summary: {text_path}")
        print(f"\n✨ Enhanced comparison complete! Check {Path('results/multi_run_comparisons').absolute()} for all files.")
        
        return report_path, csv_path, text_path

# Export enhanced report
if all_results:
    report_files = export_enhanced_comparison_report(all_results, run_configs, timing_summary, plot_path)
else:
    print("❌ No results to export")

## 6. Final Completion Logging

In [ ]:
# Notebook completion and final logging
with log_cell("Final Comparison Completion Logging"):
    total_execution_time = time.time() - notebook_start_time
    
    # Create completion summary
    completion_summary = {
        "total_runs_compared": len(selected_runs),
        "total_results_analyzed": len(all_results) if all_results else 0,
        "timing_data_available": len(run_timing_data) if run_timing_data else 0,
        "files_generated": len([f for f in [report_files] if f]) if 'report_files' in locals() else 0
    }
    
    # Log notebook completion
    log_notebook_completion(
        notebook_name="ASR Multi-Run Comparison (with Timing)",
        total_time=total_execution_time,
        results_summary=completion_summary,
        log_file="results/execution_logs/comparison_timing.json"
    )
    
    # Print execution summary
    print("\n" + "="*60)
    print("🎉 MULTI-RUN COMPARISON COMPLETED")
    print("="*60)
    
    minutes, seconds = divmod(total_execution_time, 60)
    print(f"🕐 Total comparison time: {minutes:.0f}m {seconds:.1f}s")
    print(f"📊 Runs compared: {len(selected_runs)}")
    print(f"📈 Results analyzed: {len(all_results) if all_results else 0}")
    print(f"⏱️  Timing data available: {len(run_timing_data) if run_timing_data else 0}")
    
    if 'best_overall' in locals() and best_overall is not None:
        print(f"🏆 Best performance: {best_overall['model']} ({best_overall['condition']}) - CER: {best_overall['cer']:.4f}")
    
    print(f"\n📝 Comparison timing log: results/execution_logs/comparison_timing.json")
    print(f"📁 All comparison files: results/multi_run_comparisons/")
    
    print("\n✨ Ready to analyze your ASR evaluation results with timing insights!")
    print("="*60)

## 7. Usage Instructions

### How to Use This Enhanced Notebook:

1. **Run Multiple Evaluations**: First, run the main `asr_robustness_evaluation.ipynb` multiple times
   - Each run now includes detailed timing logs
   - Timing data is automatically saved to `results/execution_logs/`

2. **Compare with Timing**: Run this notebook to:
   - Load all previous runs with timing data
   - Analyze execution time patterns
   - Compare performance vs efficiency
   - Identify bottlenecks

### New Timing Features:

- **Per-Cell Timing**: Each operation is timed and logged
- **Execution Breakdown**: See which steps take longest
- **Performance vs Efficiency**: Compare accuracy vs execution time
- **Trend Analysis**: Track timing changes across runs

### Timing Log Structure:
```json
[
  {
    "operation": "Audio Processing - Shortening Elongated Segments",
    "elapsed_seconds": 45.2,
    "elapsed_formatted": "45.2s",
    "timestamp": "2023-12-01T14:30:22"
  }
]
```

### Key Insights You Can Gain:
- **Bottleneck Identification**: Which operations slow down evaluation
- **Parameter Impact**: How settings affect execution time
- **Efficiency Optimization**: Balance between accuracy and speed
- **Resource Planning**: Estimate time for future runs

---

**🎯 Ready to compare your ASR evaluation results with detailed timing analysis!**